**This is a workup on Module 3: Target and Compound Databases**

**1. Retrieval of compounds from PubChem**

In [2]:
#Antiplamodial active compounds from PubChem
import pandas as pd
import requests
from io import StringIO

def download_pubchem_csv(aid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/CSV"
    response = requests.get(url)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        df['AID'] = aid
        return df
    else:
        print(f"Failed to fetch AID {aid}")
        return None

aids = [1828]
#Extra aids = [449703, 1619, 1828,  1815,  489011,  489016, 524796, 504696, 2195, 2196, 1822, 606570, 1159566]
dfs = [download_pubchem_csv(aid) for aid in aids if download_pubchem_csv(aid) is not None]
merged = pd.concat(dfs, ignore_index=True)


#print("Merged dataset saved as 'merged_assays.csv'")

In [3]:
merged.head()
# Save merged dataset
#merged.to_csv("merged_assays.csv", index=False)

,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Average Potency (uM),GB4 Run 1 Phenotype,...,D10 Run 2 Potency (uM),HB3 Run 1 Phenotype,HB3 Run 1 Potency (uM),HB3 Run 2 Phenotype,HB3 Run 2 Potency (uM),W2 Run 1 Phenotype,W2 Run 1 Potency (uM),W2 Run 2 Phenotype,W2 Run 2 Potency (uM),AID
0,RESULT_TYPE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FLOAT,STRING,...,FLOAT,STRING,FLOAT,STRING,FLOAT,STRING,FLOAT,STRING,FLOAT,1828
1,RESULT_DESCR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The average concentration of sample yielding h...,Indicates type of activity observed of compoun...,...,The concentration of sample yielding half-maxi...,Indicates type of activity observed of compoun...,The concentration of sample yielding half-maxi...,Indicates type of activity observed of compoun...,The concentration of sample yielding half-maxi...,Indicates type of activity observed of compoun...,The concentration of sample yielding half-maxi...,Indicates type of activity observed of compoun...,The concentration of sample yielding half-maxi...,1828
2,RESULT_UNIT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MICROMOLAR,NaN,...,MICROMOLAR,NaN,MICROMOLAR,NaN,MICROMOLAR,NaN,MICROMOLAR,NaN,MICROMOLAR,1828
3,1,26753225.0,60703.0,C1CN(CCC1CC2=CC=C(C=C2)F)CC(C3=CC=C(C=C3)Cl)O,Active,89.0,NaN,NaN,1.16521,Inhibitor,...,2.2387,Inhibitor,1.4125,Inhibitor,1.122,Inhibitor,0.2818,Inhibitor,0.2239,1828
4,2,26753246.0,2921148.0,CC1=C(C=C(C=C1)[N+](=O)[O-])S(=O)(=O)N(C)C,Inactive,0.0,NaN,NaN,NaN,Inactive,...,NaN,Inactive,NaN,Inactive,NaN,Inactive,NaN,Inactive,NaN,1828


In [4]:
import requests
import pandas as pd
import time

base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

params = {
    "target_organism": "Plasmodium falciparum",
    "limit": 1000,
    "offset": 0,
    "standard_type": "IC50",
}

all_activities = []
MAX_COMPOUNDS = 4000

while len(all_activities) < MAX_COMPOUNDS:
    print(f"Fetching molecules {params['offset']} to {params['offset'] + params['limit'] - 1}...")
    response = requests.get(base_url, params=params)
    data = response.json()

    activities = data.get("activities", [])
    if not activities:
        print("No more data available.")
        break

    all_activities.extend(activities)

    params['offset'] += params['limit']
    time.sleep(1)

all_activities = all_activities[:MAX_COMPOUNDS]

df = pd.DataFrame(all_activities)

selected_columns = [
    'molecule_chembl_id',
    'standard_value',
    'canonical_smiles',
    'ligand_efficiency',
    'standard_units',
    'standard_type'
]

df_selected = df[selected_columns]


Fetching molecules 0 to 999...
Fetching molecules 1000 to 1999...
Fetching molecules 2000 to 2999...
Fetching molecules 3000 to 3999...


In [5]:
df_selected.head()

,molecule_chembl_id,standard_value,canonical_smiles,ligand_efficiency,standard_units,standard_type
0,CHEMBL76383,73500.0,None,None,nM,IC50
1,CHEMBL77052,4.25,C[C@H]1[C@@H](OCCC2ON2C(=O)c2ccccc2O)O[C@@H]2O...,None,nM,IC50
2,CHEMBL312787,22500.0,None,None,nM,IC50
3,CHEMBL74819,200000.0,None,None,nM,IC50
4,CHEMBL307145,5660.0,Oc1cccc(O)c1O,None,nM,IC50
